# TSSS(Think Straight, Stop Smart) 기반 Multi-Hop RAG System

In [6]:
!pip install -qU langchain langchain-openai langchain-huggingface langchain-chroma sentence-transformers scikit-learn langsmith langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.9/107.9 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 283.0/283.0 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 718.4/718.4 kB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.9/234.9 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 82.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17

In [7]:
import os
import torch
from google.colab import drive, userdata


# 1. 구글 드라이브 마운트 (데이터가 드라이브에 있으므로 필수)
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 2. LangSmith 환경 변수 설정
os.environ["LANGCHAIN_API_KEY"] = userdata.get('langgrpah')
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"] = "TSSS" # 프로젝트 이름 (원하는대로 변경 가능)

print("\nLangSmith 설정 및 드라이브 마운트 완료!")


LangSmith 설정 및 드라이브 마운트 완료!


In [8]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# 설정된 경로 (사용자님 경로)
BASE_PATH = "/content/drive/MyDrive/chroma_db_bge_m3"

# 임베딩 모델 로드
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading Embedding Model on {device}...")

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True}
)

# 실제 DB 파일(chroma.sqlite3)이 있는 경로 찾기
# (압축 풀면서 폴더가 중첩되었을 경우를 대비한 사용자님의 기존 로직)
real_db_path = BASE_PATH
for root, dirs, files in os.walk(BASE_PATH):
    if "chroma.sqlite3" in files:
        real_db_path = root
        print(f"실제 DB 파일 발견 경로: {real_db_path}")
        break

# ChromaDB 로드
# 중요: collection_name을 꼭 명시해야 데이터가 보입니다!
vectorstore = Chroma(
    persist_directory=real_db_path,
    embedding_function=embedding_model,
    collection_name="multihop_rag"  # 기존 코드에 있던 컬렉션 이름
)

# 연결 확인
count = vectorstore._collection.count()
print(f"\n로드된 문서 개수: {count}개")

if count > 0:
    # 테스트 검색 (LangSmith에 기록되는지 확인용)
    test_retriever = vectorstore.as_retriever(search_kwargs={"k": 1})
    docs = test_retriever.invoke("테스트 검색")
    print(f"테스트 검색 결과: {docs[0].page_content[:50]}...")
else:
    print("여전히 데이터가 0개입니다. 경로 내의 'collection_name'이 맞는지 확인이 필요합니다.")

Loading Embedding Model on cuda...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

실제 DB 파일 발견 경로: /content/drive/MyDrive/chroma_db_bge_m3

로드된 문서 개수: 2157개
테스트 검색 결과: Title: Is Google Search better than the rest? And ...


In [10]:
import os
import torch
import json
import re
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import drive, userdata
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langsmith import traceable
from langchain_openai import ChatOpenAI

# Mdel Selection
# [Option A] OpenAI (Default)
# from langchain_openai import ChatOpenAI
# if "OPENAI_API_KEY" not in os.environ:
#     os.environ["OPENAI_API_KEY"] = userdata.get('openai')
# llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, seed=2026)
# print("LLM Selected: OpenAI (GPT-4o-mini)")

# [Option B] Google Gemini
# from langchain_google_genai import ChatGoogleGenerativeAI
# if "GOOGLE_API_KEY" not in os.environ:
#     os.environ["GOOGLE_API_KEY"] = userdata.get('gemini') # Secrets 이름 확인
# llm = ChatGoogleGenerativeAI(model="gemini-3-flash-preview", temperature=0,, seed=2026)
# print("LLM Selected: Google Gemini (1.5 Flash)")

# [Option C] OpenRouter (DeepSeek, Claude, Llama etc.)
# from langchain_openai import ChatOpenAI
# if "OPENROUTER_API_KEY" not in os.environ:
#     os.environ["OPENROUTER_API_KEY"] = userdata.get('openrouter') # Secrets 이름 확인
# llm = ChatOpenAI(
#     base_url="https://openrouter.ai/api/v1",
#     api_key=os.environ["openrouter"],
#     model="openai/gpt-oss-120b:free", # 원하는 모델명으로 변경 가능
#     temperature=0,
#     seed=2026
# )
llm = ChatOpenAI(
        model="openai/gpt-oss-120b:free",
        openai_api_key=userdata.get('openrouter'),
        openai_api_base="https://openrouter.ai/api/v1",
        temperature=0, # 사실 기반 답변
        seed=2026,
        default_headers={
            "HTTP-Referer": "https://colab.research.google.com",
            "X-Title": "MULTI HOP RAG Project"
        }
    )
print("LLM Selected: OpenRouter")


LLM Selected: OpenRouter


In [11]:
# @title TSSS (Think Straight, Stop Smart) 핵심 로직
# [핵심 기능] Stop Smart: 쿼리 유사도 계산기
def calculate_similarity(query1, query2):
    # BGE-M3로 임베딩 (GPU 사용)
    emb1 = embedding_model.embed_query(query1)
    emb2 = embedding_model.embed_query(query2)
    # 코사인 유사도 계산
    return cosine_similarity([emb1], [emb2])[0][0]

# [핵심 기능] Think Straight: 논문 Appendix B 프롬프트 생성기
def generate_tsss_prompt(main_question, history_facts):
    prompt = f"To answer the Main Question ({{ {main_question} }}), "
    if history_facts:
        prompt += "using the following facts:\n"
        for fact in history_facts:
            prompt += f"- {{ {fact} }}\n"
    prompt += "\nI propose the following additional question:\nQuestion: "
    return prompt

def extract_verbatim_evidence(query, context):
    """
    LLM에게 요약을 금지하고, 근거가 되는 '원본 문장'을 그대로 발췌하게 함
    """
    prompt = f"""
    Task: Identify the SINGLE sentence in the [Context] that acts as the core evidence for the [Query].

    [Rules]
    1. Output ONLY the raw sentence from the text.
    2. DO NOT paraphrase, summarize, or edit. Copy-paste exactly.
    3. If the answer is not in the text, output "IRRELEVANT".

    [Query]: {query}
    [Context]: {context}

    Extracted Sentence:
    """
    # 팩트 추출용 LLM 호출
    fact = llm.invoke([HumanMessage(content=prompt)]).content.strip().replace('"', '')
    return fact

def refine_answer_minimalist(question, raw_answer):
    """
    Without asking complex questions, you'll understand the LLM's intent on your own, transforming you into a police officer who best understands the situation.
    """
    prompt = f"""
    Task: Reduce the [Raw Answer] to the absolute minimum logical unit required by the [Question].

    [Guiding Principle]
    1. **Binary Choice:** Output ONLY "Yes" or "No".
    2. **Timing:** Output ONLY the time clause (e.g., "Before", "After 2023").
    3. **Entity:** Output ONLY the name/entity.
    4. **Unknowable:** If the [Raw Answer] indicates the information is missing or cannot be deduced, output EXACTLY "Insufficient information".
    5. **Formatting:** DROP ALL explanations, articles, and punctuation.

    [Input]
    Question: {question}
    Raw Answer: {raw_answer}

    Minimalist Answer:
    """

    # LLM이 스스로 판단해서 최적의 단어를 선택함
    refined = llm.invoke([HumanMessage(content=prompt)]).content.strip()

    # 혹시 모를 마침표 제거
    return refined.rstrip(".")

def check_new_terms_detected(query1, query2):
    """
    하드코딩된 정규식(대문자 체크)을 제거하고,
    단순 집합(Set) 연산으로 '새로운 의미 있는 단어'가 추가되었는지 감지합니다.
    (영어, 한국어 모두 작동)
    """
    if not query2: return True

    # 1. 전처리: 특수문자 제거 및 소문자화 (언어 중립성 확보)
    def tokenize(text):
        # 글자, 숫자, 한글 등을 제외한 특수문자는 공백으로 치환
        clean_text = re.sub(r'[^\w\s]', ' ', text).lower()
        return set(clean_text.split())

    tokens_new = tokenize(query1)
    tokens_old = tokenize(query2)

    # 2. 불용어(Stopwords) 처리 (선택 사항: 간단한 것만 제외)
    # 문법적 요소(is, the, 은, 는, 이, 가)가 아닌 '실질적 단어' 차이만 봅니다.
    # 여기서는 간단히 길이 2 이상의 단어만 체크합니다.
    diff = {w for w in (tokens_new - tokens_old) if len(w) > 1}

    if diff:
        print(f"    -> [Override] New Terms Detected: {diff}. Ignoring similarity stop.")
        return True
    return False


In [12]:
# @title 최종 retrieval
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

@traceable(name="TSSS_Final_Semantic")
def run_tsss_semantic_dedup(question, max_steps=5, threshold=0.85, top_k=5):
    print(f"\n[Processing TSSS Semantic]: {question}\n")

    history_facts = []
    past_queries = []
    evidence_list = []

    # 수집된 팩트들의 임베딩을 저장할 리스트 (비교용)
    collected_embeddings = []

    for step in range(max_steps):
        # 쿼리 생성
        prompt_text = generate_tsss_prompt(question, history_facts)
        next_query = llm.invoke([HumanMessage(content=prompt_text)]).content.strip()
        next_query = next_query.replace('"', '').replace("Question:", "").strip()

        print(f"  [Step {step+1}] Query: {next_query}")

        # 종료 조건
        if "STOP" in next_query or len(next_query) < 2:
            print("    -> LLM Stop Signal.")
            break

        is_repetitive = False
        if past_queries:
            last_query = past_queries[-1]
            sim_score = calculate_similarity(next_query, last_query)
            if sim_score > threshold:
                if check_new_terms_detected(next_query, last_query):
                    is_repetitive = False
                else:
                    print(f"    -> Loop Detected (Sim: {sim_score:.2f}). Stopping.")
                    is_repetitive = True
        if is_repetitive: break
        past_queries.append(next_query)

        # 검색
        try:
            retriever = vectorstore.as_retriever(search_kwargs={"k": top_k})
            retrieved_docs = retriever.invoke(next_query)

            if not retrieved_docs:
                print("    -> No docs found.")
                continue

            found_valid = False
            for doc in retrieved_docs:
                verbatim_fact = extract_verbatim_evidence(next_query, doc.page_content)
                if "IRRELEVANT" in verbatim_fact: continue


                # 의미 기반 중복 제거 (Semantic Deduplication)

                # 현재 찾은 팩트를 벡터로 변환
                current_emb = embedding_model.embed_query(verbatim_fact)

                is_duplicate = False

                # 기존 수집된 팩트들과 의미 유사도 비교
                if collected_embeddings:
                    # 전체 리스트와 한번에 비교 (효율적)
                    sims = cosine_similarity([current_emb], collected_embeddings)[0]
                    max_sim = np.max(sims) # 가장 비슷한 문장과의 점수

                    if max_sim > 0.90: # 90% 이상 의미가 같으면 중복 처리
                        print(f"    -> [Skip] Semantically identical content detected (Sim: {max_sim:.2f}).")
                        is_duplicate = True

                if is_duplicate:
                    continue # 다음 문서 확인
                # ---------------------------------------------------------

                source_name = doc.metadata.get('source', 'Unknown')
                print(f"    -> Found New Fact in: {source_name}")
                found_valid = True

                history_facts.append(f"Source [{source_name}]: {verbatim_fact}")

                # 임베딩 저장 (다음 비교를 위해)
                collected_embeddings.append(current_emb)

                doc_meta = doc.metadata
                evidence_list.append({
                    "author": doc_meta.get("author", "unknown"),
                    "category": doc_meta.get("category", "unknown"),
                    "fact": verbatim_fact,
                    "published_at": doc_meta.get("published_at", "unknown"),
                    "source": source_name,
                    "title": doc_meta.get("original_title", "No Title"),
                    "url": doc_meta.get("url", "")
                })
                break

            if not found_valid:
                print("    -> Docs retrieved but content irrelevant or duplicated.")

        except Exception as e:
            print(f"    -> Error: {e}")
            break

    # 최종 결과 생성 및 후처리
    print("\n[Finalizing]...")
    if not history_facts:
        final_answer = "Insufficient information" # 찾은 팩트가 아예 없으면
    else:
        final_context = "\n".join(history_facts)

        # 서술형 답변 생성
        final_prompt = f"Question: {question}\nFacts: {final_context}\nAnswer:"
        raw_answer = llm.invoke([HumanMessage(content=final_prompt)]).content.strip()

        # 단답형 정제 (여기서 'Insufficient information'이 튀어나올 수 있음)
        final_answer = refine_answer_minimalist(question, raw_answer)

    # 추론 불가면 빈값 반환
    if final_answer == "Insufficient information":
        evidence_list = []  # 리스트 강제 초기화
        print("    -> Result: Insufficient info. Cleared evidence list.")

    return {
        "answer": final_answer,
        "evidence_list": evidence_list
    }

In [ ]:
# @title 실행 테스트 코드
test_q = "Does the TechCrunch article on Twitch's subscription revenue split policy indicate a different monetization strategy compared to the TechCrunch article on Beeper's plans for Beeper Mini subscriptions?"
result_data = run_tsss_semantic_dedup(test_q, top_k=5)

print(f"\n======== FINAL JSON OUTPUT ========")
print(json.dumps(result_data, indent=2, ensure_ascii=False))


[Processing TSSS Semantic]: Does the TechCrunch article on Twitch's subscription revenue split policy indicate a different monetization strategy compared to the TechCrunch article on Beeper's plans for Beeper Mini subscriptions?

  [Step 1] Query: What are the key differences in the monetization strategies outlined in the TechCrunch articles for Twitch's subscription revenue split policy and Beeper's plans for Beeper Mini subscriptions?
    -> Found New Fact in: TechCrunch
  [Step 2] Query: What are the key differences in monetization strategies between Twitch's new subscription revenue split policy and Beeper's approach to its Mini subscriptions?
    -> [Override] New Terms Detected: {'new', 'approach', 'to', 'its', 'between'}. Ignoring similarity stop.
    -> [Skip] Semantically identical content detected (Sim: 1.00).
    -> Found New Fact in: TechCrunch
  [Step 3] Query: What are the key differences in monetization strategies between Twitch's subscription revenue split policy and B

## 평가


In [13]:
# @title dataset load
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from datasets import load_dataset

# QA 데이터셋 로드
try:
    qa_df
except NameError:
    print("QA 데이터셋 로드 중...")
    ds = load_dataset("yixuantt/MultiHopRAG", "MultiHopRAG", split="train")
    qa_df = ds.to_pandas()

# 층화 추출 (Stratified Sampling) - 비율 맞춰서 50개 뽑기
print(f"전체 데이터 개수: {len(qa_df)}")
print("Question Type별 비율에 맞춰 50개 샘플링 중...")

# sklearn을 사용하여 비율 유지하며 추출
sampled_df, _ = train_test_split(
    qa_df,
    train_size=50,
    stratify=qa_df['question_type'],
    random_state=42 # 재현성을 위해 시드 고정
)

print(f"추출된 샘플 개수: {len(sampled_df)}")
print(sampled_df['question_type'].value_counts()) # 타입별 개수 확인

QA 데이터셋 로드 중...


README.md: 0.00B [00:00, ?B/s]

MultiHopRAG.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/2556 [00:00<?, ? examples/s]

전체 데이터 개수: 2556
Question Type별 비율에 맞춰 50개 샘플링 중...
추출된 샘플 개수: 50
question_type
comparison_query    17
inference_query     16
temporal_query      11
null_query           6
Name: count, dtype: int64


In [14]:
# @title 평가 실행 (RAG Inference)
rag_answers = []
rag_evidence_lists = []

print("\n=== RAG 평가 데이터 생성 시작 (총 50개) ===")
# tqdm으로 진행 상황 바 표시
for index, row in tqdm(sampled_df.iterrows(), total=len(sampled_df), desc="Processing Queries"):
    query = row['query']

    try:
        # 기존에 만드신 함수 호출 (top_k=5 적용)
        # LangSmith 추적도 자동으로 됩니다.
        result = run_tsss_semantic_dedup(query, top_k=5)

        rag_answers.append(result['answer'])
        rag_evidence_lists.append(result['evidence_list'])

    except Exception as e:
        print(f"Error at index {index}: {e}")
        rag_answers.append("Error")
        rag_evidence_lists.append([])

# 결과 저장 및 컬럼 정리
# 원본 데이터프레임에 RAG 결과 추가
sampled_df['RAG_answer'] = rag_answers
sampled_df['RAG_evidence_list'] = rag_evidence_lists

# 요청하신 컬럼 순서대로 정리
# (데이터셋의 컬럼명이 'query', 'answer', 'evidence_list' 라고 가정)
final_eval_df = sampled_df[[
    "query",
    "evidence_list",
    "RAG_evidence_list",
    "answer",
    "RAG_answer"
]].copy()

# 5. 결과 확인 및 저장
print("\n=== 생성 완료 ===")
display(final_eval_df.head(3)) # 상위 3개 미리보기

# CSV로 저장 (필요 시)
# final_eval_df.to_csv("/content/drive/MyDrive/MultihopRAG/rag_evaluation_results.csv", index=False, encoding="utf-8-sig")
# print("결과가 'rag_evaluation_results.csv'로 저장되었습니다.")


=== RAG 평가 데이터 생성 시작 (총 50개) ===


Processing Queries:   0%|          | 0/50 [00:00<?, ?it/s]


[Processing TSSS Semantic]: Does the TechCrunch article suggest that "People's preferences regarding social media content" will shift towards uncurated experiences, while The Roar | Sports Writers Blog indicates that "Michael Cheika" values past experiences, curated or not, for preparation?

  [Step 1] Query: **Proposed additional question**

*How do the TechCrunch article’s observations about a growing user preference for uncurated, “raw” social‑media experiences compare with The Roar’s description of Michael Cheika’s attitude toward past experiences—whether curated or not—as a source of preparation and insight?*
    -> Found New Fact in: TechCrunch
  [Step 2] Query: **Answer to the Main Question**

- **TechCrunch:** The excerpt you provided (“…after everything on social media becomes so perfectly curated for us, our brains will start to crave things that are not”) clearly indicates that the author believes people’s preferences will move **away** from highly‑curated social‑media feed

Processing Queries:   2%|▏         | 1/50 [02:30<2:03:13, 150.89s/it]


[Processing TSSS Semantic]: Who is the individual targeted by Attorney General Letitia James for penalties and a business ban in New York, who also allegedly inflated the value of his Manhattan apartment, as reported by both Fortune and The Age, to conceal the diminished valuation of another one of his properties?

  [Step 1] Query: The person in question is **Michele S. Miller**.
    -> Docs retrieved but content irrelevant or duplicated.
  [Step 2] Query: **Proposed additional question**

*What is the name of the New York real‑estate developer or businessman that Attorney General Letitia James is seeking penalties and a business‑activity ban against, and who has been accused—by both Fortune and The Age—of inflating the reported value of his Manhattan apartment to hide the reduced valuation of another property?*
    -> Found New Fact in: Fortune
  [Step 3] Query: **Proposed additional question**

*What is the name of the former U.S. president who is being sued by New York Attorney Ge

Processing Queries:   4%|▍         | 2/50 [05:53<2:24:57, 181.20s/it]


[Processing TSSS Semantic]: Between the report by The Age on October 22, 2023, claiming that Google manipulates Search to maximize ad revenue, and the TechCrunch report on December 15, 2023, alleging that Google "siphons off" news publishers' content, readers, and ad revenue through anticompetitive means, was there consistency in the portrayal of Google's business practices by these news sources?

  [Step 1] Query: **Suggested additional question**

*What specific evidence or examples do The Age (Oct 22 2023) and TechCrunch (Dec 15 2023) cite to support their claims about Google’s manipulation of search results and “siphoning” of news content, and how do those details compare in tone, scope, and implied intent?*
    -> Found New Fact in: The Age
  [Step 2] Query: **Answer to the Main Question**

Yes – the two reports present a **consistent** picture of Google’s business conduct.  

| Aspect | The Age (22 Oct 2023) | TechCrunch (15 Dec 2023) |
|--------|----------------------|---------

Processing Queries:   4%|▍         | 2/50 [07:28<2:59:24, 224.25s/it]


KeyboardInterrupt: 

In [ ]:
# CSV로 저장 (필요 시)
final_eval_df.to_csv("/content/drive/MyDrive/rag_evaluation_results_oss.csv", index=False, encoding="utf-8-sig")
print("결과가 'rag_evaluation_results_oss.csv'로 저장되었습니다.")

In [4]:
# @title 평가지표
import pandas as pd
import ast
import re
import string
from collections import Counter
# import os
# from google.colab import drive, userdata


# # 1. 구글 드라이브 마운트 (데이터가 드라이브에 있으므로 필수)
# if not os.path.exists('/content/drive'):
#     drive.mount('/content/drive')

# 1. 파일 로드 (경로는 사용자 환경에 맞게 유지)
df = pd.read_csv("/content/drive/MyDrive/rag_evaluation_results_oss.csv", encoding="utf-8-sig")

# ------------------------------------------------------------------
# [Helper] 파싱 함수
# ------------------------------------------------------------------
def parse_list_robust(x):
    if not isinstance(x, str): return []
    try:
        return ast.literal_eval(x)
    except:
        pass
    fixed_str = re.sub(r'\}\s*\{', '}, {', x)
    try:
        return ast.literal_eval(fixed_str)
    except:
        return []

df['evidence_list'] = df['evidence_list'].apply(parse_list_robust)
df['RAG_evidence_list'] = df['RAG_evidence_list'].apply(parse_list_robust)

# ------------------------------------------------------------------
# [Metric 1] Retrieval Metrics (Hit, MRR, MAP)
# ------------------------------------------------------------------
def calculate_retrieval_metrics(row, k=5):
    gold_urls = set([item.get('url') for item in row['evidence_list'] if item.get('url')])
    raw_retrieved_urls = [item.get('url') for item in row['RAG_evidence_list'] if item.get('url')]

    # 중복 제거 (순서 유지)
    retrieved_urls = []
    seen = set()
    for url in raw_retrieved_urls:
        if url not in seen:
            retrieved_urls.append(url)
            seen.add(url)
    retrieved_urls = retrieved_urls[:k]

    # Hit@K
    hit = 1 if not gold_urls.isdisjoint(retrieved_urls) else 0

    # MRR@K
    mrr = 0
    for i, url in enumerate(retrieved_urls):
        if url in gold_urls:
            mrr = 1 / (i + 1)
            break

    # MAP@K
    num_gold = len(gold_urls)
    if num_gold == 0:
        ap = 0
    else:
        hits = 0
        sum_precisions = 0
        for i, url in enumerate(retrieved_urls):
            if url in gold_urls:
                hits += 1
                sum_precisions += hits / (i + 1)
        ap = sum_precisions / num_gold

    return pd.Series([hit, mrr, ap], index=[f'Hit@{k}', f'MRR@{k}', f'MAP@{k}'])

# [Metric 2] Answer Match Accuracy (Exact Match & F1)
def normalize_answer(s):
    """
    평가를 위해 답변 텍스트를 정규화합니다.
    (소문자 변환, 문장부호 제거)
    """

    def white_space_fix(text):
        return ' '.join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)

    def lower(text):
        return str(text).lower()

    if not s or pd.isna(s): return ""

    return white_space_fix(remove_punc(lower(s)))

def calculate_qa_metrics(row):
    """
    정답(Answer)과 모델 예측(RAG_Answer)을 비교합니다.
    """
    gold_text = normalize_answer(row['answer'])
    pred_text = normalize_answer(row['RAG_answer'])

    # Exact Match (EM): 완전히 일치하는가? (0 or 1)
    em = 1 if gold_text == pred_text else 0


    return pd.Series([em], index=['Exact Match Accuracy'])

# ------------------------------------------------------------------
# [Execution] 전체 적용 및 저장
# ------------------------------------------------------------------
K_VALUE = 5

# 1. Retrieval Score 계산
retrieval_scores = df.apply(lambda row: calculate_retrieval_metrics(row, k=K_VALUE), axis=1)

# 2. QA Score 계산 (추가된 부분)
qa_scores = df.apply(calculate_qa_metrics, axis=1)

# 3. 전체 데이터프레임 병합
final_df = pd.concat([df, retrieval_scores, qa_scores], axis=1)

# 결과 출력 (평균 점수 확인)
print(f"======== Evaluation Results (K={K_VALUE}) ========")
# 백분율(%)로 변환하여 출력
summary = final_df[[f'Hit@{K_VALUE}', f'MRR@{K_VALUE}', f'MAP@{K_VALUE}', 'Exact Match Accuracy']].mean() * 100
print(summary)

# CSV 저장
save_path = "/content/drive/MyDrive/rag_evaluation_with_all_metrics_oss.csv"
final_df.to_csv(save_path, index=False, encoding="utf-8-sig")
print(f"\n[Done] 결과 파일이 저장되었습니다: {save_path}")

======== Evaluation Results (K=5) ========
Hit@5                   68.000000
MRR@5                   59.166667
MAP@5                   31.291667
Exact Match Accuracy    64.000000
dtype: float64

[Done] 결과 파일이 저장되었습니다: /content/drive/MyDrive/rag_evaluation_with_all_metrics.csv


In [5]:
pd.read_csv("/content/drive/MyDrive/rag_evaluation_with_all_metrics.csv")

,query,evidence_list,RAG_evidence_list,answer,RAG_answer,Hit@5,MRR@5,MAP@5,Exact Match Accuracy
0,"Does the TechCrunch article suggest that ""Peop...","[{'author': 'Sarah Perez', 'category': 'techno...","[{'author': 'Christy Doran', 'category': 'spor...",Yes,Yes,1.0,1.000000,0.500000,1
1,Who is the individual targeted by Attorney Gen...,"[{'author': 'Michael R. Sisak, The Associated ...","[{'author': 'Michael R. Sisak, The Associated ...",Donald Trump,Donald Trump,1.0,1.000000,0.500000,1
2,"Between the report by The Age on October 22, 2...","[{'author': 'Kyle Wiggers', 'category': 'techn...","[{'author': 'Sarah Perez', 'category': 'techno...",Yes,Yes,1.0,1.000000,0.333333,1
3,Who is the individual under 30 who was once co...,"[{'author': 'Elizabeth Lopatto', 'category': '...","[{'author': 'Elizabeth Lopatto', 'category': '...",Sam Bankman-Fried,Sam Bankman-Fried,1.0,1.000000,0.250000,1
4,Does the TechCrunch article report on new hiri...,"[{'author': 'Alyssa Stringer', 'category': 'te...","[{'author': 'Jessica Conditt', 'category': 'te...",no,Yes,1.0,1.000000,0.500000,0
5,Does the TechCrunch article discussing Meta's ...,"[{'author': 'Morgan Sung', 'category': 'techno...","[{'author': 'Morgan Sung', 'category': 'techno...",Yes,Yes,1.0,1.000000,0.500000,1
6,"Which company, according to Eddy Cue, had no v...","[{'author': 'David Pierce', 'category': 'techn...","[{'author': 'Sarah Perez', 'category': 'techno...",Google,Google,1.0,1.000000,0.666667,1
7,Between the article from 'The Independent - Li...,"[{'author': 'Chelsea Ritschel', 'category': 'e...","[{'author': 'Amber Raiken', 'category': 'enter...",Yes,Yes,1.0,1.000000,1.000000,1
8,Considering the information from an article in...,[],[],Insufficient information.,Insufficient information,0.0,0.000000,0.000000,1
9,Does the article from 'The Independent - Life ...,"[{'author': 'Chelsea Ritschel', 'category': 'e...","[{'author': 'Chelsea Ritschel', 'category': 'e...",Yes,Yes,1.0,1.000000,0.500000,1


여기서부터는 코드 실행 x

## 실험 코드

In [ ]:
import os
import torch
import json
import re
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import drive, userdata
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# ==============================================================================
# 1. LLM 모델 선택 섹션 (원하는 모델의 주석을 해제하여 사용하세요)
# ==============================================================================

# [Option A] OpenAI (Default)
from langchain_openai import ChatOpenAI
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = userdata.get('openai')
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, seed=2026)
print("LLM Selected: OpenAI (GPT-4o-mini)")

# [Option B] Google Gemini
# from langchain_google_genai import ChatGoogleGenerativeAI
# if "GOOGLE_API_KEY" not in os.environ:
#     os.environ["GOOGLE_API_KEY"] = userdata.get('gemini') # Secrets 이름 확인
# llm = ChatGoogleGenerativeAI(model="gemini-3-flash-preview", temperature=0,, seed=2026)
# print("LLM Selected: Google Gemini (1.5 Flash)")

# [Option C] OpenRouter (DeepSeek, Claude, Llama etc.)
# from langchain_openai import ChatOpenAI
# if "OPENROUTER_API_KEY" not in os.environ:
#     os.environ["OPENROUTER_API_KEY"] = userdata.get('openrouter') # Secrets 이름 확인
# llm = ChatOpenAI(
#     base_url="https://openrouter.ai/api/v1",
#     api_key=os.environ["openrouter"],
#     model="openai/gpt-oss-120b:free", # 원하는 모델명으로 변경 가능
#     temperature=0,
#     seed=2026
# )
# print("LLM Selected: OpenRouter")


# TSSS (Think Straight, Stop Smart) 핵심 로직


# [핵심 기능] Stop Smart: 쿼리 유사도 계산기
def calculate_similarity(query1, query2):
    # BGE-M3로 임베딩 (GPU 사용)
    emb1 = embedding_model.embed_query(query1)
    emb2 = embedding_model.embed_query(query2)
    # 코사인 유사도 계산
    return cosine_similarity([emb1], [emb2])[0][0]

# [핵심 기능] Think Straight: 논문 Appendix B 프롬프트 생성기
def generate_tsss_prompt(main_question, history_facts):
    prompt = f"To answer the Main Question ({{ {main_question} }}), "
    if history_facts:
        prompt += "using the following facts:\n"
        for fact in history_facts:
            prompt += f"- {{ {fact} }}\n"
    prompt += "\nI propose the following additional question:\nQuestion: "
    return prompt

# # [메인 로직] TSSS 실행 함수
# def run_tsss_pipeline(question, max_steps=5, threshold=0.85):
#     print(f"\n[Processing TSSS]: {question}\n")

#     history_facts = []
#     past_queries = []
#     collected_docs = []

#     for step in range(max_steps):
#         # --- Step 1. 다음 쿼리 생성 ---
#         prompt_text = generate_tsss_prompt(question, history_facts)
#         msg = HumanMessage(content=prompt_text)

#         # LLM 호출
#         next_query = llm.invoke([msg]).content.strip()
#         next_query = next_query.replace('"', '').replace("Question:", "").strip()

#         print(f"  [Step {step+1}] Generated Query: {next_query}")

#         # --- Step 2. 종료 조건 검사 (Stop Smart) ---
#         # A. LLM 종료 신호
#         if "STOP" in next_query or len(next_query) < 3:
#             print("    -> [Terminator]: LLM signaled STOP.")
#             break

#         # B. 유사도 기반 강제 종료
#         is_repetitive = False
#         for past_q in past_queries:
#             sim_score = calculate_similarity(next_query, past_q)
#             if sim_score > threshold:
#                 print(f"    -> [Terminator]: Similarity {sim_score:.2f} > {threshold}. STOPPING loop.")
#                 is_repetitive = True
#                 break

#         if is_repetitive:
#             break

#         past_queries.append(next_query)

#         # --- Step 3. 검색 (Retrieval) ---
#         try:
#             retrieved_docs = vectorstore.similarity_search(next_query, k=5)

#             if not retrieved_docs:
#                 print("    -> No documents found.")
#                 continue

#             top_doc = retrieved_docs[0]
#             collected_docs.append(top_doc)
#             print(f"    -> Retrieved: {top_doc.metadata.get('source', 'Unknown')}")

#             # --- Step 4. 사실 추출 (Fact Generation) ---
#             fact_extraction_prompt = f"""
#             Based on the Context, briefly answer the Sub-Question.
#             Sub-Question: {next_query}
#             Context: {top_doc.page_content}

#             Answer (Fact):
#             """
#             fact_response = llm.invoke([HumanMessage(content=fact_extraction_prompt)]).content.strip()
#             history_facts.append(fact_response)

#         except Exception as e:
#             print(f"    -> Error during search: {e}")
#             break

#     # --- Step 5. 최종 답변 생성 ---
#     print("\n[Final Generation]...")

#     if not history_facts:
#         final_context = "No relevant information found."
#     else:
#         final_context = "\n".join([f"- {fact}" for fact in history_facts])

#     final_prompt = f"""
#     Based on the collected facts below, provide a clear and concise answer to the Main Question.

#     Main Question: {question}

#     Collected Facts:
#     {final_context}

#     Final Answer:
#     """

#     final_answer = llm.invoke([HumanMessage(content=final_prompt)]).content.strip()
#     return final_answer

def extract_verbatim_evidence(query, context):
    """
    LLM에게 요약을 금지하고, 근거가 되는 '원본 문장'을 그대로 발췌하게 함
    """
    prompt = f"""
    Task: Identify the SINGLE sentence in the [Context] that acts as the core evidence for the [Query].

    [Rules]
    1. Output ONLY the raw sentence from the text.
    2. DO NOT paraphrase, summarize, or edit. Copy-paste exactly.
    3. If the answer is not in the text, output "IRRELEVANT".

    [Query]: {query}
    [Context]: {context}

    Extracted Sentence:
    """
    # 팩트 추출용 LLM 호출
    fact = llm.invoke([HumanMessage(content=prompt)]).content.strip().replace('"', '')
    return fact

# ------------------------------------------------------------------
# [Helper 2] 답변 정제기 (Answer Refiner) - 핵심 추가 기능
# ------------------------------------------------------------------
def refine_to_simplest_form(question, raw_answer):
    """
    서술형 답변을 질문 타입에 맞는 단답형(Entity)으로 변환
    """
    prompt = f"""
    You are a "Data Extractor". Convert the [Raw Answer] into the simplest possible format based on the [Question].

    [Rules]
    1. Who -> Output ONLY the Name .
    2. When -> Output ONLY the Date or Year.
    3. Yes/No -> Output ONLY "Yes" or "No".
    4. What/Which -> Output ONLY the Entity name.
    5. REMOVE all filler phrases like "The answer is...", "Based on the context...", periods(.), etc.

    [Question]: {question}
    [Raw Answer]: {raw_answer}

    Simplest Answer:
    """
    refined = llm.invoke([HumanMessage(content=prompt)]).content.strip()
    return refined

# ------------------------------------------------------------------
# [메인 로직] TSSS + 포맷팅 + 정제기
# ------------------------------------------------------------------
def run_tsss_final_formatted(question, max_steps=5, threshold=0.85):
    print(f"\n[Processing TSSS]: {question}\n")

    history_facts = []
    past_queries = []
    evidence_list = []

    for step in range(max_steps):
        # --- Step 1. 다음 쿼리 생성 (Think Straight) ---
        prompt_text = generate_tsss_prompt(question, history_facts)
        next_query = llm.invoke([HumanMessage(content=prompt_text)]).content.strip()
        next_query = next_query.replace('"', '').replace("Question:", "").strip()

        print(f"  [Step {step+1}] Query: {next_query}")

        # --- Step 2. 종료 조건 (Stop Smart) ---
        if "STOP" in next_query or len(next_query) < 3:
            print("    -> LLM Stop Signal.")
            break

        is_repetitive = False
        for past_q in past_queries:
            sim_score = calculate_similarity(next_query, past_q)
            if sim_score > threshold:
                print(f"    -> Loop Detected (Sim: {sim_score:.2f}). Stopping.")
                is_repetitive = True
                break
        if is_repetitive: break
        past_queries.append(next_query)

        # --- Step 3. 검색 ---
        try:
            retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

            # invoke를 쓰면 LangSmith가 "아, 검색을 수행했구나" 하고 문서를 기록함
            retrieved_docs = retriever.invoke(next_query)

            if not retrieved_docs:
                print("    -> No docs found.")
                continue

            top_doc = retrieved_docs[0]

            # --- Step 4. 원문 Fact (Evidence) 추출 ---
            # 여기서 추출한 문장이 최종 evidence_list의 'fact'가 됩니다.
            verbatim_fact = extract_verbatim_evidence(next_query, top_doc.page_content)

            if "IRRELEVANT" in verbatim_fact:
                print("    -> Found doc, but irrelevant content.")
                continue

            print(f"    -> Found Evidence in: {top_doc.metadata.get('source')}")

            # --- Step 5. TSSS 문맥 업데이트 (History) ---
            # 다음 질문 생성을 위한 짧은 요약 (Evidence가 아님)
            short_fact = f"Fact: {verbatim_fact}"
            history_facts.append(short_fact)

            # --- Step 6. Evidence List 구성 ---
            doc_meta = top_doc.metadata
            evidence_list.append({
                    "author": doc_meta.get("author", "unknown"),
                    "category": doc_meta.get("category", "unknown"),
                    "fact": verbatim_fact,
                    "published_at": doc_meta.get("published_at", "unknown"),
                    "source": doc_meta.get("source","unknowmn"),
                    "title": doc_meta.get("original_title", "No Title"),
                    "url": doc_meta.get("url", "")
                })
            break

            # 중복 방지 후 추가
            if verbatim_fact not in [e['fact'] for e in evidence_list]:
                evidence_list.append(evidence_item)

        except Exception as e:
            print(f"    -> Error: {e}")
            break

    # --- Step 7. 최종 답변 생성 및 정제 ---
    print("\n[Finalizing]...")
    if not history_facts:
        final_answer = "N/A"
    else:
        # 1. 서술형 답변 생성 (Raw)
        final_context = "\n".join(history_facts)
        final_prompt = f"""
        Based on the facts below, answer the Question.
        Question: {question}
        Facts: {final_context}
        """
        raw_answer = llm.invoke([HumanMessage(content=final_prompt)]).content.strip()

        # 2. 답변 정제 (Refinement -> Simplest Form)
        final_answer = refine_to_simplest_form(question, raw_answer)

    # --- Step 8. 최종 JSON 반환 ---
    return {
        "answer": final_answer,
        "evidence_list": evidence_list
    }


LLM Selected: OpenAI (GPT-4o-mini)


In [ ]:
# test
test_q = "Who is the individual associated with the cryptocurrency industry facing a criminal trial on fraud and conspiracy charges, as reported by both The Verge and TechCrunch, and is accused by prosecutors of committing fraud for personal gain?"
# result = run_tsss_pipeline(test_q)
# print(f"\n======== RESULT ========\n{result}")
result_data = run_tsss_final_formatted(test_q)

print(f"\n======== FINAL JSON OUTPUT ========")
print(json.dumps(result_data, indent=2, ensure_ascii=False))


[Processing TSSS]: Who is the individual associated with the cryptocurrency industry facing a criminal trial on fraud and conspiracy charges, as reported by both The Verge and TechCrunch, and is accused by prosecutors of committing fraud for personal gain?

  [Step 1] Query: Who is the individual facing legal issues in the cryptocurrency sector, specifically related to allegations of fraud and conspiracy, as highlighted by major tech news outlets?
    -> Found Evidence in: The Verge

[Finalizing]...

======== FINAL JSON OUTPUT ========
{
  "answer": "Sam Bankman-Fried",
  "evidence_list": [
    {
      "author": "Elizabeth Lopatto",
      "category": "technology",
      "fact": "Bankman-Fried’s defense can also introduce risks for people who dealt with him.",
      "published_at": "2023-09-28 12:00:00.000000000Z",
      "source": "The Verge",
      "title": "The FTX trial is bigger than Sam Bankman-Fried",
      "url": "https://www.theverge.com/2023/9/28/23893269/ftx-sam-bankman-fried

In [ ]:
# @title 정밀 탐색 추가
import os
import re
import json
import torch
from langsmith import traceable
from langchain_core.messages import HumanMessage
from langchain_chroma import Chroma
from sklearn.metrics.pairwise import cosine_similarity

# ------------------------------------------------------------------
# 1. [Refactored] 언어 중립적 키워드 감지기 (No Regex Hard-coding)
# ------------------------------------------------------------------
def check_new_terms_detected(query1, query2):
    """
    하드코딩된 정규식(대문자 체크)을 제거하고,
    단순 집합(Set) 연산으로 '새로운 의미 있는 단어'가 추가되었는지 감지합니다.
    (영어, 한국어 모두 작동)
    """
    if not query2: return True

    # 1. 전처리: 특수문자 제거 및 소문자화 (언어 중립성 확보)
    def tokenize(text):
        # 글자, 숫자, 한글 등을 제외한 특수문자는 공백으로 치환
        clean_text = re.sub(r'[^\w\s]', ' ', text).lower()
        return set(clean_text.split())

    tokens_new = tokenize(query1)
    tokens_old = tokenize(query2)

    # 2. 불용어(Stopwords) 처리 (선택 사항: 간단한 것만 제외)
    # 문법적 요소(is, the, 은, 는, 이, 가)가 아닌 '실질적 단어' 차이만 봅니다.
    # 여기서는 간단히 길이 2 이상의 단어만 체크합니다.
    diff = {w for w in (tokens_new - tokens_old) if len(w) > 1}

    if diff:
        print(f"    -> [Override] New Terms Detected: {diff}. Ignoring similarity stop.")
        return True
    return False

# ------------------------------------------------------------------
# 2. [Refactored] 프롬프트 일반화 (No Rule-based Hard-coding)
# ------------------------------------------------------------------
def refine_answer_general(question, raw_answer):
    """
    'Who', 'When' 같은 특정 단어 규칙을 제거하고,
    질문의 의도에 맞는 '핵심 엔티티'만 추출하도록 일반화합니다.
    """
    prompt = f"""
    You are a "Core Entity Extractor".
    Extract the specific Value, Name, Date, or Entity that directly answers the [Question] from the [Raw Answer].

    [Rules]
    1. Output ONLY the core entity. No sentences.
    2. Remove all filler words (e.g., "The answer is", "It is").
    3. If multiple entities are required, list them with commas.

    [Question]: {question}
    [Raw Answer]: {raw_answer}

    Core Entity:
    """
    return llm.invoke([HumanMessage(content=prompt)]).content.strip()

def refine_answer_minimalist(question, raw_answer):
    """
    Without asking complex questions, you'll understand the LLM's intent on your own, transforming you into a police officer who best understands the situation.
    """
    prompt = f"""
    Task: Reduce the [Raw Answer] to the absolute minimum logical unit required by the [Question].

    [Guiding Principle]
    1. **Binary Choice:** Output ONLY "Yes" or "No".
    2. **Timing:** Output ONLY the time clause (e.g., "Before", "After 2023").
    3. **Entity:** Output ONLY the name/entity.
    4. **Unknowable:** If the [Raw Answer] indicates the information is missing or cannot be deduced, output EXACTLY "Insufficient information".
    5. **Formatting:** DROP ALL explanations, articles, and punctuation.

    [Input]
    Question: {question}
    Raw Answer: {raw_answer}

    Minimalist Answer:
    """

    # LLM이 스스로 판단해서 최적의 단어를 선택함
    refined = llm.invoke([HumanMessage(content=prompt)]).content.strip()

    # 혹시 모를 마침표 제거
    return refined.rstrip(".")

# ------------------------------------------------------------------
# 3. [Refactored] 메인 파이프라인 (Fully Parametrized)
# ------------------------------------------------------------------
@traceable(name="TSSS_Final_Clean")
def run_tsss_clean(question, max_steps=5, threshold=0.85, top_k=5): # top_k 파라미터화
    print(f"\n[Processing TSSS Clean]: {question}\n")

    history_facts = []
    past_queries = []
    evidence_list = []

    for step in range(max_steps):
        # --- Step 1. 쿼리 생성 ---
        # (기존 generate_tsss_prompt함수 재사용)
        prompt_text = generate_tsss_prompt(question, history_facts)
        next_query = llm.invoke([HumanMessage(content=prompt_text)]).content.strip()
        next_query = next_query.replace('"', '').replace("Question:", "").strip()

        print(f"  [Step {step+1}] Query: {next_query}")

        # --- Step 2. 종료 조건 (유사도 + 신규 단어 체크) ---
        if "STOP" in next_query or len(next_query) < 2:
            print("    -> LLM Stop Signal.")
            break

        is_repetitive = False
        if past_queries:
            last_query = past_queries[-1]
            sim_score = calculate_similarity(next_query, last_query)

            # 유사도가 높아도, 새로운 단어(check_new_terms_detected)가 있으면 진행
            if sim_score > threshold:
                if check_new_terms_detected(next_query, last_query):
                    is_repetitive = False
                else:
                    print(f"    -> Loop Detected (Sim: {sim_score:.2f}). Stopping.")
                    is_repetitive = True

        if is_repetitive: break
        past_queries.append(next_query)

        # --- Step 3. 검색 (파라미터 top_k 사용) ---
        try:
            retriever = vectorstore.as_retriever(search_kwargs={"k": top_k})
            retrieved_docs = retriever.invoke(next_query)

            if not retrieved_docs:
                print("    -> No docs found.")
                continue

            found_valid = False
            for doc in retrieved_docs:
                verbatim_fact = extract_verbatim_evidence(next_query, doc.page_content)

                if "IRRELEVANT" in verbatim_fact: continue

                source_name = doc.metadata.get('source', 'Unknown')
                print(f"    -> Found Evidence in: {source_name}")
                found_valid = True

                history_facts.append(f"Source [{source_name}]: {verbatim_fact}")

                # 메타데이터 안전하게 가져오기 (.get 기본값 활용)
                doc_meta = doc.metadata
                evidence_list.append({
                    "author": doc_meta.get("author", "unknown"),
                    "category": doc_meta.get("category", "unknown"),
                    "fact": verbatim_fact,
                    "published_at": doc_meta.get("published_at", "unknown"),
                    "source": source_name,
                    "title": doc_meta.get("original_title", "No Title"),
                    "url": doc_meta.get("url", "")
                })
                break

            if not found_valid:
                print("    -> Docs retrieved but content irrelevant.")

        except Exception as e:
            print(f"    -> Error: {e}")
            break

    # --- Step 4. 최종 결과 ---
    print("\n[Finalizing]...")
    if not history_facts:
        final_answer = "N/A"
    else:
        final_context = "\n".join(history_facts)
        final_prompt = f"Question: {question}\nFacts: {final_context}\nAnswer:"
        raw_answer = llm.invoke([HumanMessage(content=final_prompt)]).content.strip()
        final_answer = refine_answer_general(question, raw_answer)

    return {
        "answer": final_answer,
        "evidence_list": evidence_list
    }

# # 실행
# test_q = "Who is the individual associated with the cryptocurrency industry facing a criminal trial on fraud and conspiracy charges, as reported by both The Verge and TechCrunch?"
# result_data = run_tsss_clean(test_q, top_k=3) # 파라미터로 제어

# print(f"\n======== FINAL JSON OUTPUT ========")
# print(json.dumps(result_data, indent=2, ensure_ascii=False))

In [ ]:
# 실행
test_q = "Who is the individual associated with the cryptocurrency industry facing a criminal trial on fraud and conspiracy charges, as reported by both The Verge and TechCrunch?"
result_data = run_tsss_clean(test_q, top_k=5) # 파라미터로 제어

print(f"\n======== FINAL JSON OUTPUT ========")
print(json.dumps(result_data, indent=2, ensure_ascii=False))


[Processing TSSS Clean]: Who is the individual associated with the cryptocurrency industry facing a criminal trial on fraud and conspiracy charges, as reported by both The Verge and TechCrunch?

  [Step 1] Query: Who is the individual facing legal issues in the cryptocurrency sector, specifically related to fraud and conspiracy allegations, as highlighted by The Verge and TechCrunch?
    -> Found Evidence in: The Verge
  [Step 2] Query: Who is Bankman-Fried and what are the implications of his defense strategy for others involved in the cryptocurrency industry?
    -> Found Evidence in: TechCrunch
  [Step 3] Query: Who is Bankman-Fried and what are the specific charges he is facing in the criminal trial?
    -> Found Evidence in: TechCrunch
  [Step 4] Query: What are the specific charges that Bankman-Fried is facing in his criminal trial?
    -> [Override] New Terms Detected: {'that', 'his'}. Ignoring similarity stop.
    -> Found Evidence in: TechCrunch
  [Step 5] Query: What are the

In [ ]:
# @title 중복 제거
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

@traceable(name="TSSS_Final_Semantic")
def run_tsss_semantic_dedup(question, max_steps=5, threshold=0.85, top_k=5):
    print(f"\n[Processing TSSS Semantic]: {question}\n")

    history_facts = []
    past_queries = []
    evidence_list = []

    # [New] 수집된 팩트들의 임베딩을 저장할 리스트 (비교용)
    collected_embeddings = []

    for step in range(max_steps):
        # 1. 쿼리 생성
        prompt_text = generate_tsss_prompt(question, history_facts)
        next_query = llm.invoke([HumanMessage(content=prompt_text)]).content.strip()
        next_query = next_query.replace('"', '').replace("Question:", "").strip()

        print(f"  [Step {step+1}] Query: {next_query}")

        # 2. 종료 조건
        if "STOP" in next_query or len(next_query) < 2:
            print("    -> LLM Stop Signal.")
            break

        is_repetitive = False
        if past_queries:
            last_query = past_queries[-1]
            sim_score = calculate_similarity(next_query, last_query)
            if sim_score > threshold:
                if check_new_terms_detected(next_query, last_query):
                    is_repetitive = False
                else:
                    print(f"    -> Loop Detected (Sim: {sim_score:.2f}). Stopping.")
                    is_repetitive = True
        if is_repetitive: break
        past_queries.append(next_query)

        # 3. 검색
        try:
            retriever = vectorstore.as_retriever(search_kwargs={"k": top_k})
            retrieved_docs = retriever.invoke(next_query)

            if not retrieved_docs:
                print("    -> No docs found.")
                continue

            found_valid = False
            for doc in retrieved_docs:
                verbatim_fact = extract_verbatim_evidence(next_query, doc.page_content)
                if "IRRELEVANT" in verbatim_fact: continue

                # ---------------------------------------------------------
                # [New Logic] 의미 기반 중복 제거 (Semantic Deduplication)
                # ---------------------------------------------------------
                # 1. 현재 찾은 팩트를 벡터로 변환
                current_emb = embedding_model.embed_query(verbatim_fact)

                is_duplicate = False

                # 2. 기존 수집된 팩트들과 의미 유사도 비교
                if collected_embeddings:
                    # 전체 리스트와 한번에 비교 (효율적)
                    sims = cosine_similarity([current_emb], collected_embeddings)[0]
                    max_sim = np.max(sims) # 가장 비슷한 문장과의 점수

                    if max_sim > 0.90: # 90% 이상 의미가 같으면 중복 처리
                        print(f"    -> [Skip] Semantically identical content detected (Sim: {max_sim:.2f}).")
                        is_duplicate = True

                if is_duplicate:
                    continue # 다음 문서 확인
                # ---------------------------------------------------------

                source_name = doc.metadata.get('source', 'Unknown')
                print(f"    -> Found New Fact in: {source_name}")
                found_valid = True

                history_facts.append(f"Source [{source_name}]: {verbatim_fact}")

                # 임베딩 저장 (다음 비교를 위해)
                collected_embeddings.append(current_emb)

                doc_meta = doc.metadata
                evidence_list.append({
                    "author": doc_meta.get("author", "unknown"),
                    "category": doc_meta.get("category", "unknown"),
                    "fact": verbatim_fact,
                    "published_at": doc_meta.get("published_at", "unknown"),
                    "source": source_name,
                    "title": doc_meta.get("original_title", "No Title"),
                    "url": doc_meta.get("url", "")
                })
                break

            if not found_valid:
                print("    -> Docs retrieved but content irrelevant or duplicated.")

        except Exception as e:
            print(f"    -> Error: {e}")
            break

    # 4. 최종 결과
    print("\n[Finalizing]...")
    if not history_facts:
        final_answer = "N/A"
    else:
        final_context = "\n".join(history_facts)
        final_prompt = f"Question: {question}\nFacts: {final_context}\nAnswer:"
        raw_answer = llm.invoke([HumanMessage(content=final_prompt)]).content.strip()
        final_answer = refine_answer_general(question, raw_answer)

    return {
        "answer": final_answer,
        "evidence_list": evidence_list
    }


In [ ]:
# 실행 테스트
test_q = "Who is the individual associated with the cryptocurrency industry facing a criminal trial on fraud and conspiracy charges, as reported by both The Verge and TechCrunch?"
result_data = run_tsss_semantic_dedup(test_q, top_k=5)

print(f"\n======== FINAL JSON OUTPUT ========")
print(json.dumps(result_data, indent=2, ensure_ascii=False))


[Processing TSSS Semantic]: Who is the individual associated with the cryptocurrency industry facing a criminal trial on fraud and conspiracy charges, as reported by both The Verge and TechCrunch?

  [Step 1] Query: Who is the individual facing legal issues in the cryptocurrency sector, specifically related to fraud and conspiracy allegations, as highlighted by The Verge and TechCrunch?
    -> Found New Fact in: The Verge
  [Step 2] Query: Who is Bankman-Fried and what are the implications of his defense strategy for others involved in the cryptocurrency industry?
    -> Found New Fact in: TechCrunch
  [Step 3] Query: Who is Bankman-Fried and what are the specific charges he is facing in the criminal trial?
    -> Found New Fact in: TechCrunch
  [Step 4] Query: What are the specific charges that Bankman-Fried is facing in his criminal trial?
    -> [Override] New Terms Detected: {'that', 'his'}. Ignoring similarity stop.
    -> [Skip] Semantically identical content detected (Sim: 1.00

In [ ]:
# @title 중복 제거
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

@traceable(name="TSSS_Final_Semantic")
def run_tsss_semantic_dedup(question, max_steps=5, threshold=0.85, top_k=5):
    print(f"\n[Processing TSSS Semantic]: {question}\n")

    history_facts = []
    past_queries = []
    evidence_list = []

    # [New] 수집된 팩트들의 임베딩을 저장할 리스트 (비교용)
    collected_embeddings = []

    for step in range(max_steps):
        # 1. 쿼리 생성
        prompt_text = generate_tsss_prompt(question, history_facts)
        next_query = llm.invoke([HumanMessage(content=prompt_text)]).content.strip()
        next_query = next_query.replace('"', '').replace("Question:", "").strip()

        print(f"  [Step {step+1}] Query: {next_query}")

        # 2. 종료 조건
        if "STOP" in next_query or len(next_query) < 2:
            print("    -> LLM Stop Signal.")
            break

        is_repetitive = False
        if past_queries:
            last_query = past_queries[-1]
            sim_score = calculate_similarity(next_query, last_query)
            if sim_score > threshold:
                if check_new_terms_detected(next_query, last_query):
                    is_repetitive = False
                else:
                    print(f"    -> Loop Detected (Sim: {sim_score:.2f}). Stopping.")
                    is_repetitive = True
        if is_repetitive: break
        past_queries.append(next_query)

        # 3. 검색
        try:
            retriever = vectorstore.as_retriever(search_kwargs={"k": top_k})
            retrieved_docs = retriever.invoke(next_query)

            if not retrieved_docs:
                print("    -> No docs found.")
                continue

            found_valid = False
            for doc in retrieved_docs:
                verbatim_fact = extract_verbatim_evidence(next_query, doc.page_content)
                if "IRRELEVANT" in verbatim_fact: continue

                # ---------------------------------------------------------
                # [New Logic] 의미 기반 중복 제거 (Semantic Deduplication)
                # ---------------------------------------------------------
                # 1. 현재 찾은 팩트를 벡터로 변환
                current_emb = embedding_model.embed_query(verbatim_fact)

                is_duplicate = False

                # 2. 기존 수집된 팩트들과 의미 유사도 비교
                if collected_embeddings:
                    # 전체 리스트와 한번에 비교 (효율적)
                    sims = cosine_similarity([current_emb], collected_embeddings)[0]
                    max_sim = np.max(sims) # 가장 비슷한 문장과의 점수

                    if max_sim > 0.90: # 90% 이상 의미가 같으면 중복 처리
                        print(f"    -> [Skip] Semantically identical content detected (Sim: {max_sim:.2f}).")
                        is_duplicate = True

                if is_duplicate:
                    continue # 다음 문서 확인
                # ---------------------------------------------------------

                source_name = doc.metadata.get('source', 'Unknown')
                print(f"    -> Found New Fact in: {source_name}")
                found_valid = True

                history_facts.append(f"Source [{source_name}]: {verbatim_fact}")

                # 임베딩 저장 (다음 비교를 위해)
                collected_embeddings.append(current_emb)

                doc_meta = doc.metadata
                evidence_list.append({
                    "author": doc_meta.get("author", "unknown"),
                    "category": doc_meta.get("category", "unknown"),
                    "fact": verbatim_fact,
                    "published_at": doc_meta.get("published_at", "unknown"),
                    "source": source_name,
                    "title": doc_meta.get("original_title", "No Title"),
                    "url": doc_meta.get("url", "")
                })
                break

            if not found_valid:
                print("    -> Docs retrieved but content irrelevant or duplicated.")

        except Exception as e:
            print(f"    -> Error: {e}")
            break

    # 4. 최종 결과
    print("\n[Finalizing]...")
    if not history_facts:
        final_answer = "N/A"
    else:
        final_context = "\n".join(history_facts)
        final_prompt = f"Question: {question}\nFacts: {final_context}\nAnswer:"
        raw_answer = llm.invoke([HumanMessage(content=final_prompt)]).content.strip()
        final_answer = refine_answer_general(question, raw_answer)

    return {
        "answer": final_answer,
        "evidence_list": evidence_list
    }


In [ ]:
# 실행 테스트
test_q = "Does the TechCrunch article on Twitch's subscription revenue split policy indicate a different monetization strategy compared to the TechCrunch article on Beeper's plans for Beeper Mini subscriptions?"
result_data = run_tsss_semantic_dedup(test_q, top_k=5)

print(f"\n======== FINAL JSON OUTPUT ========")
print(json.dumps(result_data, indent=2, ensure_ascii=False))


[Processing TSSS Semantic]: Does the TechCrunch article on Twitch's subscription revenue split policy indicate a different monetization strategy compared to the TechCrunch article on Beeper's plans for Beeper Mini subscriptions?

  [Step 1] Query: What are the key differences in the monetization strategies outlined in the TechCrunch articles for Twitch's subscription revenue split policy and Beeper's plans for Beeper Mini subscriptions?
    -> Found New Fact in: TechCrunch
  [Step 2] Query: What are the key differences in monetization strategies between Twitch's new subscription revenue split policy and Beeper's approach to its Mini subscriptions?
    -> [Override] New Terms Detected: {'between', 'approach', 'its', 'to', 'new'}. Ignoring similarity stop.
    -> [Skip] Semantically identical content detected (Sim: 1.00).
    -> Found New Fact in: TechCrunch
  [Step 3] Query: What are the key differences in monetization strategies between Twitch's subscription revenue split policy and B

In [ ]:
# @title 단답형 대답 추가
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

@traceable(name="TSSS_Final_Semantic")
def run_tsss_semantic_dedup(question, max_steps=5, threshold=0.85, top_k=5):
    print(f"\n[Processing TSSS Semantic]: {question}\n")

    history_facts = []
    past_queries = []
    evidence_list = []

    # [New] 수집된 팩트들의 임베딩을 저장할 리스트 (비교용)
    collected_embeddings = []

    for step in range(max_steps):
        # 1. 쿼리 생성
        prompt_text = generate_tsss_prompt(question, history_facts)
        next_query = llm.invoke([HumanMessage(content=prompt_text)]).content.strip()
        next_query = next_query.replace('"', '').replace("Question:", "").strip()

        print(f"  [Step {step+1}] Query: {next_query}")

        # 2. 종료 조건
        if "STOP" in next_query or len(next_query) < 2:
            print("    -> LLM Stop Signal.")
            break

        is_repetitive = False
        if past_queries:
            last_query = past_queries[-1]
            sim_score = calculate_similarity(next_query, last_query)
            if sim_score > threshold:
                if check_new_terms_detected(next_query, last_query):
                    is_repetitive = False
                else:
                    print(f"    -> Loop Detected (Sim: {sim_score:.2f}). Stopping.")
                    is_repetitive = True
        if is_repetitive: break
        past_queries.append(next_query)

        # 3. 검색
        try:
            retriever = vectorstore.as_retriever(search_kwargs={"k": top_k})
            retrieved_docs = retriever.invoke(next_query)

            if not retrieved_docs:
                print("    -> No docs found.")
                continue

            found_valid = False
            for doc in retrieved_docs:
                verbatim_fact = extract_verbatim_evidence(next_query, doc.page_content)
                if "IRRELEVANT" in verbatim_fact: continue

                # ---------------------------------------------------------
                # [New Logic] 의미 기반 중복 제거 (Semantic Deduplication)
                # ---------------------------------------------------------
                # 1. 현재 찾은 팩트를 벡터로 변환
                current_emb = embedding_model.embed_query(verbatim_fact)

                is_duplicate = False

                # 2. 기존 수집된 팩트들과 의미 유사도 비교
                if collected_embeddings:
                    # 전체 리스트와 한번에 비교 (효율적)
                    sims = cosine_similarity([current_emb], collected_embeddings)[0]
                    max_sim = np.max(sims) # 가장 비슷한 문장과의 점수

                    if max_sim > 0.90: # 90% 이상 의미가 같으면 중복 처리
                        print(f"    -> [Skip] Semantically identical content detected (Sim: {max_sim:.2f}).")
                        is_duplicate = True

                if is_duplicate:
                    continue # 다음 문서 확인
                # ---------------------------------------------------------

                source_name = doc.metadata.get('source', 'Unknown')
                print(f"    -> Found New Fact in: {source_name}")
                found_valid = True

                history_facts.append(f"Source [{source_name}]: {verbatim_fact}")

                # 임베딩 저장 (다음 비교를 위해)
                collected_embeddings.append(current_emb)

                doc_meta = doc.metadata
                evidence_list.append({
                    "author": doc_meta.get("author", "unknown"),
                    "category": doc_meta.get("category", "unknown"),
                    "fact": verbatim_fact,
                    "published_at": doc_meta.get("published_at", "unknown"),
                    "source": source_name,
                    "title": doc_meta.get("original_title", "No Title"),
                    "url": doc_meta.get("url", "")
                })
                break

            if not found_valid:
                print("    -> Docs retrieved but content irrelevant or duplicated.")

        except Exception as e:
            print(f"    -> Error: {e}")
            break

    # 4. 최종 결과
    # print("\n[Finalizing]...")
    # if not history_facts:
    #     final_answer = "N/A"
    # else:
    #     final_context = "\n".join(history_facts)
    #     final_prompt = f"Question: {question}\nFacts: {final_context}\nAnswer:"
    #     raw_answer = llm.invoke([HumanMessage(content=final_prompt)]).content.strip()
    #     # refine_answer_general -> refine_answer_minimalist
    #     final_answer = refine_answer_minimalist(question, raw_answer)

    # return {
    #     "answer": final_answer,
    #     "evidence_list": evidence_list
    # }
    # ... (run_tsss_semantic_dedup 함수의 마지막 부분) ...

    # 4. 최종 결과 생성 및 후처리
    print("\n[Finalizing]...")
    if not history_facts:
        final_answer = "Insufficient information" # 찾은 팩트가 아예 없으면
    else:
        final_context = "\n".join(history_facts)

        # 서술형 답변 생성
        final_prompt = f"Question: {question}\nFacts: {final_context}\nAnswer:"
        raw_answer = llm.invoke([HumanMessage(content=final_prompt)]).content.strip()

        # 단답형 정제 (여기서 'Insufficient information'이 튀어나올 수 있음)
        final_answer = refine_answer_minimalist(question, raw_answer)

    # 추론 불가면 빈값 반환
    if final_answer == "Insufficient information":
        evidence_list = []  # 리스트 강제 초기화
        print("    -> Result: Insufficient info. Cleared evidence list.")

    return {
        "answer": final_answer,
        "evidence_list": evidence_list
    }

In [ ]:
# 실행 테스트
test_q = "Does the TechCrunch article on Twitch's subscription revenue split policy indicate a different monetization strategy compared to the TechCrunch article on Beeper's plans for Beeper Mini subscriptions?"
result_data = run_tsss_semantic_dedup(test_q, top_k=5)

print(f"\n======== FINAL JSON OUTPUT ========")
print(json.dumps(result_data, indent=2, ensure_ascii=False))


[Processing TSSS Semantic]: Does the TechCrunch article on Twitch's subscription revenue split policy indicate a different monetization strategy compared to the TechCrunch article on Beeper's plans for Beeper Mini subscriptions?

  [Step 1] Query: What are the key differences in the monetization strategies outlined in the TechCrunch articles for Twitch's subscription revenue split policy and Beeper's plans for Beeper Mini subscriptions?
    -> Found New Fact in: TechCrunch
  [Step 2] Query: What are the key differences in monetization strategies between Twitch's new subscription revenue split policy and Beeper's approach to its Mini subscriptions?
    -> [Override] New Terms Detected: {'between', 'approach', 'its', 'to', 'new'}. Ignoring similarity stop.
    -> [Skip] Semantically identical content detected (Sim: 1.00).
    -> Found New Fact in: TechCrunch
  [Step 3] Query: What are the key differences in monetization strategies between Twitch's subscription revenue split policy and B

In [ ]:
# 실행 테스트
test_q = "Does the Sporting News article suggest that streaming services do not require a subscription for viewing the Cowboys vs. 49ers game, in contrast to the Polygon article's claim about film availability on streaming platforms without a subscription?"
result_data = run_tsss_semantic_dedup(test_q, top_k=5)

print(f"\n======== FINAL JSON OUTPUT ========")
print(json.dumps(result_data, indent=2, ensure_ascii=False))



[Processing TSSS Semantic]: Does the Sporting News article suggest that streaming services do not require a subscription for viewing the Cowboys vs. 49ers game, in contrast to the Polygon article's claim about film availability on streaming platforms without a subscription?

  [Step 1] Query: What specific details or statements from the Sporting News article indicate whether streaming services require a subscription for viewing the Cowboys vs. 49ers game, and how do these details compare to the claims made in the Polygon article regarding film availability on streaming platforms?
    -> Found New Fact in: Sporting News
  [Step 2] Query: Does the Sporting News article indicate that viewers can access the Cowboys vs. 49ers game for free through any streaming service, or does it emphasize the need for a subscription or trial?
    -> [Override] New Terms Detected: {'does', 'through', 'any', 'trial', 'service', 'need', 'access', 'emphasize', 'that', 'can', 'free', 'it', 'viewers'}. Ignorin

In [ ]:
# 실행 테스트
test_q = "Has the advice provided by Sporting News to bettors regarding the evaluation of betting opportunities and offers involved reading requirements, going with the favored Eagles, and focusing on hype between the reports published on September 28, 2023, and December 18, 2023?"
result_data = run_tsss_semantic_dedup(test_q, top_k=5)

print(f"\n======== FINAL JSON OUTPUT ========")
print(json.dumps(result_data, indent=2, ensure_ascii=False))



[Processing TSSS Semantic]: Has the advice provided by Sporting News to bettors regarding the evaluation of betting opportunities and offers involved reading requirements, going with the favored Eagles, and focusing on hype between the reports published on September 28, 2023, and December 18, 2023?

  [Step 1] Query: What specific strategies or criteria does Sporting News recommend for evaluating betting opportunities, particularly in relation to the favored Eagles, and how do these strategies evolve between the reports published on September 28, 2023, and December 18, 2023?
    -> Found New Fact in: Sporting News
  [Step 2] Query: What specific strategies or considerations should bettors keep in mind when evaluating betting opportunities, particularly in relation to favored teams like the Eagles, based on the advice from Sporting News?
    -> [Override] New Terms Detected: {'considerations', 'teams', 'keep', 'bettors', 'should', 'mind', 'advice', 'like', 'when', 'from', 'based'}. Ign

In [ ]:
# 실행 테스트
test_q = "Which company, covered by Engadget and Polygon, is set to release an updated gaming hardware with over 300 improvements on November 16, emphasizing a singular performance target for developers?"
result_data = run_tsss_semantic_dedup(test_q, top_k=5)

print(f"\n======== FINAL JSON OUTPUT ========")
print(json.dumps(result_data, indent=2, ensure_ascii=False))



[Processing TSSS Semantic]: Which company, covered by Engadget and Polygon, is set to release an updated gaming hardware with over 300 improvements on November 16, emphasizing a singular performance target for developers?

  [Step 1] Query: What specific features or improvements can gamers expect from the updated gaming hardware being released on November 16?
    -> Found New Fact in: Engadget
  [Step 2] Query: What are some of the key features and improvements of the Steam Deck OLED that enhance its performance for gamers?
    -> Found New Fact in: Polygon
  [Step 3] Query: What are the key features and improvements of the Steam Deck OLED that enhance its gaming performance and user experience?
    -> [Override] New Terms Detected: {'gaming', 'experience', 'user'}. Ignoring similarity stop.
    -> [Skip] Semantically identical content detected (Sim: 1.00).
    -> Found New Fact in: Polygon
  [Step 4] Query: What specific features and improvements does the Steam Deck OLED offer compar

In [ ]:
# 실행 테스트
test_q = "Considering the information from an article by The Verge and another by Forbes about Sygic, which letter represents both the first character of the European country where Sygic is headquartered and the last character of the name of Sygic's CEO as mentioned in these articles?"
result_data = run_tsss_semantic_dedup(test_q, top_k=5)

print(f"\n======== FINAL JSON OUTPUT ========")
print(json.dumps(result_data, indent=2, ensure_ascii=False))



[Processing TSSS Semantic]: Considering the information from an article by The Verge and another by Forbes about Sygic, which letter represents both the first character of the European country where Sygic is headquartered and the last character of the name of Sygic's CEO as mentioned in these articles?

  [Step 1] Query: What is the name of the European country where Sygic is headquartered, and who is the CEO of Sygic?
    -> Docs retrieved but content irrelevant or duplicated.
  [Step 2] Query: What is the name of the European country where Sygic is headquartered, and who is the CEO of Sygic?
    -> Loop Detected (Sim: 1.00). Stopping.

[Finalizing]...
    -> Result: Insufficient info. Cleared evidence list.

======== FINAL JSON OUTPUT ========
{
  "answer": "Insufficient information",
  "evidence_list": []
}


In [ ]:
# import pandas as pd

# # 1. DB에 있는 데이터 중 1개만 샘플로 가져오기
# # (limit=1로 설정하여 전체를 다 긁어오지 않고 빠르게 확인)
# sample_data = vectorstore._collection.get(limit=1)

# # 2. 데이터가 있는지 확인
# if len(sample_data['ids']) == 0:
#     print("❌ 데이터베이스가 비어있습니다! 경로를 다시 확인해주세요.")
# else:
#     print("✅ 데이터베이스 연결 성공! 샘플 데이터를 분석합니다.\n")

#     # 3. 메타데이터(Metadata) 구조 확인
#     metadata = sample_data['metadatas'][0]
#     print(f"📌 [Metadata Keys]: {list(metadata.keys())}")
#     print("-" * 50)

#     # 4. 보기 좋게 출력 (Pandas DataFrame 활용)
#     # 실제 값들이 어떻게 들어있는지 예시를 보여줍니다.
#     df = pd.DataFrame([metadata])
#     display(df) # Colab에서는 display()가 표를 예쁘게 보여줍니다.

#     print("\n" + "-" * 50)
#     print("📄 [Document Content (Content Snippet)]:")
#     # 본문 내용은 너무 길 수 있으니 앞부분 200자만 출력
#     content_preview = sample_data['documents'][0][:200]
#     print(f"{content_preview}...")
#     print("-" * 50)

#     # 5. 전체 문서 개수 확인
#     total_count = vectorstore._collection.count()
#     print(f"\n📊 총 저장된 문서 개수: {total_count}개")

✅ 데이터베이스 연결 성공! 샘플 데이터를 분석합니다.

📌 [Metadata Keys]: ['category', 'doc_id', 'author', 'url', 'source', 'chunk_index', 'original_title', 'published_at']
--------------------------------------------------


,category,doc_id,author,url,source,chunk_index,original_title,published_at
0,entertainment,0,None,https://mashable.com/article/cyber-monday-deal...,Mashable,0,200+ of the best deals from Amazon's Cyber Mon...,2023-11-27 08:45:59.000000000Z



--------------------------------------------------
📄 [Document Content (Content Snippet)]:
Title: 200+ of the best deals from Amazon's Cyber Monday sale | Date: 2023-11-27 08:45:59.000000000Z | Source: Mashable

Table of Contents Table of Contents Echo, Fire TV, and Kindle deals Apple deals...
--------------------------------------------------

📊 총 저장된 문서 개수: 2157개
